# 03 - Text Test

This notebook is the first functional checkpoint for a customer sentiment triage use case. It validates text-only GPT-4o behavior before adding image or audio signals.

Pre-requisite: run `02_configure.ipynb` first.

---

## Step 1 – Load environment variables

In [ ]:
import json
import os
from pathlib import Path

import requests
from azure.identity import AzureCliCredential
from dotenv import load_dotenv

load_dotenv(dotenv_path="../env/.env")

endpoint = os.environ["AZURE_AI_ENDPOINT"].rstrip("/")
deployment = os.environ["AZURE_AI_DEPLOYMENT"]
auth_mode = os.environ.get("AZURE_AUTH_MODE", "aad").lower()
if auth_mode != "aad":
    raise RuntimeError(f"Unsupported auth mode for this demo: {auth_mode}. Expected 'aad'.")

credential = AzureCliCredential()
credential.get_token("https://cognitiveservices.azure.com/.default")
credential_source = "AzureCliCredential"

output_dir = Path("../outputs")
output_dir.mkdir(exist_ok=True)

print(f"Endpoint          : {endpoint}")
print(f"Deployment        : {deployment}")
print(f"Credential source : {credential_source}")

Endpoint          : https://aigpt4o01mqdk5.cognitiveservices.azure.com
Deployment        : gpt-4o
Credential source : AzureCliCredential


: 

## Step 2 – Create clients / connections

Initialise any SDK clients using the loaded environment variables.

In [5]:
def chat_completion(messages, max_completion_tokens=256):
    # Keep request construction explicit so learners can map payload to API behavior.
    url = f"{endpoint}/openai/deployments/{deployment}/chat/completions?api-version=2024-10-21"
    payload = {"messages": messages, "max_completion_tokens": max_completion_tokens}
    token = credential.get_token("https://cognitiveservices.azure.com/.default").token
    response = requests.post(
        url,
        headers={"Authorization": f"Bearer {token}", "Content-Type": "application/json"},
        json=payload,
        timeout=60,
    )
    if response.status_code >= 400:
        raise RuntimeError(f"Request failed: {response.status_code} {response.text}")
    body = response.json()
    text = body["choices"][0]["message"]["content"]
    return text, body

print("Helper ready")

Helper ready


## Step 3 – Baseline test

A minimal call to confirm the deployment is reachable.

In [ ]:
messages = [
    {"role": "system", "content": "You are a concise assistant for sentiment triage validation."},
    {"role": "user", "content": "A customer wrote: 'My order is still missing and no one has replied for 3 days.' In one sentence, classify the sentiment and urgency."},
]

text_response, raw_response = chat_completion(messages)
print(text_response)

output_path = output_dir / "03_text_test_result.json"
output_path.write_text(json.dumps(raw_response, indent=2), encoding="utf-8")
print(f"Saved: {output_path.resolve()}")

Teams stage multimodal testing to identify and address specific issues systematically, reduce complexity, and optimize resources for better accuracy and control.
Saved: C:\Users\hannahhowell\OneDrive - Microsoft\Documents\Git\azure-solution-prototypes\demos\gpt4o01-text-image-audio\outputs\03_text_test_result.json


## Step 4 – Demo scenario

Implement the key scenario that validates the customer hypothesis.

In [15]:
messages = [
    {"role": "system", "content": "You are a concise sentiment analysis assistant."},
    {
        "role": "user",
        "content": "Customer message: I have contacted support three times about my delayed order and still have no clear update. I needed this item for an event this weekend, and now I am frustrated and considering canceling. Please provide a brief sentiment summary with sentiment label, confidence, and one-line rationale.",
    },
]

text_response_2, raw_response_2 = chat_completion(messages)
print(text_response_2)

output_path_2 = output_dir / "03_text_test_second_prompt.json"
output_path_2.write_text(json.dumps(raw_response_2, indent=2), encoding="utf-8")
print(f"Saved: {output_path_2.resolve()}")

**Sentiment Summary:** Negative  
**Confidence:** High  
**Rationale:** The customer expresses frustration due to repeated unresolved support contacts and the impact on their upcoming event, indicating dissatisfaction and a potential intent to cancel.
Saved: C:\Users\hannahhowell\OneDrive - Microsoft\Documents\Git\azure-solution-prototypes\demos\gpt4o01-text-image-audio\outputs\03_text_test_second_prompt.json


---

## Text checkpoint complete

Next: run **04_image_test.ipynb** for local-image input validation.